In [ ]:
from utils import get_dataset_lines

def get_mass_table(toy=False):
    # Reads the integer mass table and returns a dictionary mapping mass to amino acid
    mass_table = {}
    with open('integer_mass_table.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            aa = parts[0]
            mass = int(parts[1])
            if mass not in mass_table:
                mass_table[mass] = aa
    
    if toy:
        mass_table[4] = 'X'
        mass_table[5] = 'Z'
                
    return mass_table

def get_aa_to_mass_table(toy=False):
    # Reads the integer mass table and returns a dictionary mapping amino acid to mass
    aa_map = {}
    with open('integer_mass_table.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            aa = parts[0]
            mass = int(parts[1])
            aa_map[aa] = mass
            
    if toy:
        aa_map['X'] = 4
        aa_map['Z'] = 5
    return aa_map

def peptide_mass(peptide, aa_mass_map):
    # Computes the total mass of a peptide string
    mass = 0
    for aa in peptide:
        mass += aa_mass_map[aa]
    return mass

def score_peptide(peptide, spectrum, aa_mass_map):
    # Calculates the score of a peptide against a spectrum
    # The spectrum list is 0-indexed where index i corresponds to mass i+1
    mass = 0
    score = 0
    for aa in peptide:
        mass += aa_mass_map[aa]
        if 0 < mass <= len(spectrum):
            score += spectrum[mass - 1]
    return score

def parse_psm_search_input(lines):
    # Parses input lines for the PSM Search Problem
    lines = [l.strip() for l in lines if l.strip()]
    if len(lines) < 3:
        return [], "", 0
        
    threshold = int(lines[-1])
    proteome = lines[-2]
    spectral_vectors = []
    for line in lines[:-2]:
        spectral_vectors.append(list(map(int, line.split())))
        
    return spectral_vectors, proteome, threshold

def parse_spectral_dictionary_input(lines):
    # Parses input lines for the Size/Probability of Spectral Dictionary Problems
    lines = [l.strip() for l in lines if l.strip()]
    if len(lines) < 3:
        return [], 0, 0
    
    spectrum = list(map(int, lines[0].split()))
    threshold = int(lines[1])
    max_score = int(lines[2])
    return spectrum, threshold, max_score

def parse_spectral_alignment_input(lines):
    # Parses input lines for the Spectral Alignment Problem
    lines = [l.strip() for l in lines if l.strip()]
    if len(lines) < 3:
        return "", [], 0
    peptide = lines[0]
    spectrum = list(map(int, lines[1].split()))
    k = int(lines[2])
    return peptide, spectrum, k

# The Peptide Identification Problem

If you followed our struggles to sequence antibiotic peptides, then you will agree that we should be wary of jumping to the conclusion that the highest-scoring peptide for $DinosaurSpectrum$ must have generated this spectrum.

Despite many attempts, researchers have still not devised a scoring function that reliably assigns the highest score to the biologically correct peptide, i.e., the peptide that generated the spectrum. Fortunately, although the correct peptide often does not achieve the highest score among all peptides, it typically does score highest among all peptides limited to the species’s proteome. As a result, we can transition from peptide sequencing to peptide identification by limiting our search to peptides present in the proteome, which we concatenate into a single amino acid string $Proteome$.

**Peptide Identification Problem**: Find a peptide from a proteome with maximum score against a spectrum.

**Input**: A spectral vector $Spectrum'$ and an amino acid string $Proteome$.

**Output**: An amino acid string $Peptide$ that maximizes $Score(Peptide', Spectrum')$ among all substrings of $Proteome$.

**Code Challenge**: Solve the Peptide Identification Problem.

**Input**: A space-delimited spectral vector $Spectrum'$ and an amino acid string $Proteome$.

**Output**: A substring of $Proteome$ with maximum score against $Spectrum'$.

**Sample Input**:

```
0 0 0 4 -2 -3 -1 -7 6 5 3 2 1 9 3 -8 0 3 1 2 1 8
XZZXZXXXZXZZXZXXZ
```

**Sample Output**:

```
ZXZXX
```

In [ ]:
def PeptideIdentification(spectrum, proteome, aa_mass_map):
    # Find the peptide in the proteome that maximizes the score against the given spectrum
    target_mass = len(spectrum)
    max_score = -float('inf')
    best_peptide = ""
    
    n = len(proteome)
    
    for i in range(n):
        current_mass = 0
        current_score = 0
        for j in range(i, n):
            aa = proteome[j]
            if aa not in aa_mass_map:
                break
                
            w = aa_mass_map[aa]
            current_mass += w
            
            if current_mass > target_mass:
                break
                
            # Accumulate score based on the mass of the current prefix
            current_score += spectrum[current_mass - 1]
            
            # Check if current substring matches target mass and update maximum score
            if current_mass == target_mass:
                if current_score > max_score:
                    max_score = current_score
                    best_peptide = proteome[i : j+1]
                break
                
    return best_peptide

In [ ]:
### Sample Input Execution
spectrum_str = "0 0 0 4 -2 -3 -1 -7 6 5 3 2 1 9 3 -8 0 3 1 2 1 8"
proteome = "XZZXZXXXZXZZXZXXZ"

spectrum = list(map(int, spectrum_str.split()))

aa_mass_map = get_aa_to_mass_table(toy=True)

result = PeptideIdentification(spectrum, proteome, aa_mass_map)
print(result)

### Verification
expected_output = "ZXZXX"
assert result == expected_output, f"Expected {expected_output}, but got {result}"
print("Test passed!")

ZXZXX
Test passed!


In [ ]:
### Dataset Test
test_dataset_filename = 'dataset_30270_2.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    if lines:
        spectrum_line = lines[0].strip()
        proteome_line = lines[1].strip()
        
        spectrum = list(map(int, spectrum_line.split()))
        proteome = proteome_line
        aa_mass_map = get_aa_to_mass_table(toy=False)
        
        result = PeptideIdentification(spectrum, proteome, aa_mass_map)
        print(result)
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

IHRAAGSNSSETFDT


# Searching for peptide-spectrum matches

Like peptide sequencing algorithms, peptide identification algorithms may return an erroneous peptide, particularly if the score of the highest-scoring peptide found in the proteome is much lower than the score of the highest-scoring peptide over all peptides. For this reason, biologists usually establish a score threshold and only pay attention to a solution of the Peptide Identification Problem if its score is at least equal to the threshold.

Given a set of spectral vectors $SpectralVectors$, an amino acid string $Proteome$, and a score threshold $threshold$, we will solve the Peptide Identification Problem for each vector $Spectrum'$ in $SpectralVectors$ and identify a peptide $Peptide$ having maximum score for this spectral vector over all peptides in $Proteome$ (ties are broken arbitrarily). If $Score(Peptide', Spectrum')$ is greater than or equal to $threshold$, then we conclude that $Peptide$ is present in the sample and call the pair ($Peptide$, $Spectrum'$) a peptide-spectrum match (PSM). The resulting collection of PSMs for $SpectralVectors$ is denoted $PSM_{threshold}(Proteome, SpectralVectors)$.

**PSM Search Problem**: Identify all peptide-spectrum matches scoring above a threshold for a set of spectra and a proteome.

**Input**: A set of spectral vectors $SpectralVectors$, an amino acid string $Proteome$, and an integer $threshold$.

**Output**: The set $PSM_{threshold}(Proteome, SpectralVectors)$.

The following pseudocode solves the PSM Search Problem using an algorithm that you just implemented to solve the Peptide identification Problem, which we call `PeptideIdentification`.

```plaintext
PSMSearch(SpectralVectors, Proteome, threshold)
    PSMSet ← an empty set
    for each vector Spectrum' in SpectralVectors
        Peptide ← PeptideIdentification(Spectrum', Proteome)
        if Score(Peptide, Spectrum') ≥ threshold
            add the PSM (Peptide, Spectrum') to PSMSet 
    return PSMSet
```

**Code Challenge**: Implement `PSMSearch` to solve the Peptide Search Problem.

**Input**: A set of space-delimited spectral vectors $SpectralVectors$, an amino acid string $Proteome$, and an integer $threshold$.

**Output**: The set $PSM_{threshold}(Proteome, SpectralVectors)$.

**Sample Input**:

```
-1 5 -4 5 3 -1 -4 5 -1 0 0 4 -1 0 1 4 4 4
-4 2 -2 -4 4 -5 -1 4 -1 2 5 -3 -1 3 2 -3
XXXZXZXXZXZXXXZXXZX
5 
```

**Sample Output**:

```
XZXZ
```

In [ ]:
def PeptideIdentification(spectrum, proteome, aa_mass_map):
    # Find the peptide in the proteome that maximizes the score against the given spectrum
    target_mass = len(spectrum)
    max_score = -float('inf')
    best_peptide = ""
    
    n = len(proteome)
    
    for i in range(n):
        current_mass = 0
        current_score = 0
        for j in range(i, n):
            aa = proteome[j]
            if aa not in aa_mass_map:
                break
                
            w = aa_mass_map[aa]
            current_mass += w
            
            if current_mass > target_mass:
                break
                
            # Accumulate score based on the mass of the current prefix
            current_score += spectrum[current_mass - 1]
            
            # Check if current substring matches target mass and update maximum score
            if current_mass == target_mass:
                if current_score > max_score:
                    max_score = current_score
                    best_peptide = proteome[i : j+1]
                break
                
    return best_peptide

def PSMSearch(spectral_vectors, proteome, threshold, aa_mass_map):
    # Identifies all Peptide-Spectrum Matches matching the threshold
    psm_set = set()
    for spectrum in spectral_vectors:
        peptide = PeptideIdentification(spectrum, proteome, aa_mass_map)
        if not peptide:
            continue
            
        score = score_peptide(peptide, spectrum, aa_mass_map)
        
        if score >= threshold:
            psm_set.add(peptide)
            
    return psm_set

In [ ]:
### Sample Input Execution
sample_input = """
-1 5 -4 5 3 -1 -4 5 -1 0 0 4 -1 0 1 4 4 4
-4 2 -2 -4 4 -5 -1 4 -1 2 5 -3 -1 3 2 -3
XXXZXZXXZXZXXXZXXZX
5
""".strip().split('\n')

spectral_vectors, proteome, threshold = parse_psm_search_input(sample_input)
aa_mass_map = get_aa_to_mass_table(toy=True)

psm_set = PSMSearch(spectral_vectors, proteome, threshold, aa_mass_map)

for psm in psm_set:
    print(psm)

### Verification
expected = {"XZXZ"}
assert psm_set == expected, f"Expected {expected}, but got {psm_set}"
print("Sample test passed!")

XZXZ
Sample test passed!


In [ ]:
### Dataset Test
dataset_filename = 'dataset_30270_7.txt'

try:
    lines = get_dataset_lines(dataset_filename)
    if lines:
        spectral_vectors, proteome, threshold = parse_psm_search_input(lines)
        
        aa_mass_map = get_aa_to_mass_table(toy=False)
        
        psm_set = PSMSearch(spectral_vectors, proteome, threshold, aa_mass_map)
        
        for psm in psm_set:
            print(psm)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")

MDVVAKIQCRLHAN
IYSMRDHWFQYEI
HNGHPTWNNLWVD
ILMFRDMIRACRDN
MDHKKIETSQTLYAT
LYRQYSEQFMVNA
KRAGSGDGCGYGVQVE
LDAEYWTVHFAMKV
TVFKAGWTHTLCRC
VKTGMEIMEPDEGHA
AIEPTSAMSFYWSINA
CWTNVYVLTRKRAD
VDIVKLMTNCENDLGH
LKCKKPYPTPWGDICA
ADASGCNPIADPVHANK
WPDLYQIQEQHMTG
KWHDCMDQNVRC
KADFIYGLSIRSGQ
MKEKFAQLWWNGVY
CPANECYGGHIFMFVL
ENDYVWPKRIAAAN
CGADWMLKADFIY
EHWVVSLTVPHYPDM


# Size of Spectral Dictionary Problem

We will first compute the number of peptides in a spectral dictionary, since this simpler problem will provide insights on how to compute the probability of a spectral dictionary.

**Size of Spectral Dictionary Problem**: Find the size of the spectral dictionary for a given spectral vector and score threshold.

**Input**: A spectral vector $Spectrum'$ and an integer $threshold$.

**Output**: The number of peptides in $Dictionary_{threshold}(Spectrum')$.

We will use dynamic programming to solve the Size of Spectral Dictionary Problem. Given a spectral vector $Spectrum' = (s_1, \ldots, s_m)$, we define its $i$-prefix (for $i$ between 1 and $m$) as $Spectrum_i = (s_1, \ldots, s_i)$ and introduce a variable $Size(i, t)$ as the number of peptides $Peptide$ of mass $i$ such that $Score(Peptide, Spectrum'_i)$ is equal to $t$. 

For example, consider the spectral vector $Spectrum' = (4, -3, -2, 3, 3, -4, 5, -3, -1, -1, 3, 4, 1, 3)$ of length 14 and the toy amino acid alphabet consisting of amino acids $X$ and $Z$ with respective masses 4 and 5. There are only three peptides of mass 13 ($XXZ$, $XZX$, and $ZXX$); the first two peptides have score 1 against $Spectrum'_{13}$, and the third has score 3. Thus, $Size(13, 1) = 2$, $Size(13, 3) = 1$, and $Size(13, t) = 0$ for all values of $t$ other than 1 and 3.

The key to establishing a recurrence relation for computing $Size(i, t)$ is to realize that the set of peptides contributing to $Size(i, t)$ can be split into 20 subsets depending on their final amino acid $a$. Each peptide ending in a specific amino acid $a$ results in a shorter peptide with mass $i - |a|$ and score $t - s_i$ if we remove $a$ from the peptide (here, $|a|$ denotes the mass of $a$). Thus, as illustrated in the figure below:

$$Size(i, t) = \sum_{\text{all amino acids } a} Size(i - |a|, t - s_i)$$

It should be noted that starting the rows ($t$-index) at index 0 might be conceptually simpler to understand, but leads to wrong results if the score dips into negative territory, i.e., correctness depends on the values in the spectral vector. For example, if the spectral vector in the example above were changed from:

`4, -3, -2, 3, 3, -4, 5, -3, -1, -1, 3, 4, 1, 3` 

to

`4, -3, -2, 3, 3, -4, 5, -3, -4, -4, 3, 4, 1, 6`,

an intermediate score of -1 appears in columns 9 and 10, which is not representable in the matrix as defined above. (You can try this out in your own code for the code challenge on the next page).

The same is true for how far into the positive $t$-index the matrix extends, which can be more than $t$ from $Size(i, t)$. The code challenge on the next page supplies a `max_score` parameter to define the maximum $t$-index in the matrix for this purpose. A safe value for `max_score` would be:

$$max\_score = \sum_{s_i > 0} s_i$$

and similarly a safe value for the minimum $t$-index would be:

$$min\_score = \sum_{s_i < 0} s_I$$

**Code Challenge**: Solve the Size of Spectral Dictionary Problem.

**Input**: A spectral vector $Spectrum'$, an integer $threshold$, and an integer $max\_score$.

**Output**: The size of the dictionary $Dictionary_{threshold}(Spectrum')$.

Note: Use the 20 amino acid alphabet as well as the provided `max_score` for the height of your table. Your answer should be the number of peptides whose score is at least $T$ and at most $max\_score$.

**Sample Input**:

```
4 -3 -2 3 3 -4 5 -3 -1 -1 3 4 1 3
1
8
```

**Sample Output**:

```
3
```

In [ ]:
from collections import defaultdict

def SizeOfSpectralDictionary(spectrum, threshold, max_score, aa_mass_map):
    # Computes the size of the spectral dictionary using dynamic programming
    m = len(spectrum)
    # dp[i][t] = number of peptides of mass i with score t
    dp = [defaultdict(int) for _ in range(m + 1)]
    
    # Base case: mass 0 has score 0 with count 1
    dp[0][0] = 1
    
    # Get all masses (including duplicates for I/L, K/Q) to iterate over
    # We must iterate over every amino acid type, not just unique masses.
    aa_masses = list(aa_mass_map.values())
    
    for i in range(1, m + 1):
        for w in aa_masses:
            if i >= w:
                prev_mass = i - w
                current_score_gain = spectrum[i-1]
                
                for prev_score, count in dp[prev_mass].items():
                    new_score = prev_score + current_score_gain
                    
                    if 0 <= new_score <= max_score:
                        dp[i][new_score] += count
                    
    # Sum counts for peptides of full mass m with score in valid range
    result = 0
    for score in dp[m]:
        if threshold <= score <= max_score:
            result += dp[m][score]
            
    return result

In [ ]:
### Sample Input Execution
sample_input = """
4 -3 -2 3 3 -4 5 -3 -1 -1 3 4 1 3
1
8
""".strip().split('\n')

spectrum, threshold, max_score = parse_spectral_dictionary_input(sample_input)
aa_mass_map = get_aa_to_mass_table(toy=True)

result = SizeOfSpectralDictionary(spectrum, threshold, max_score, aa_mass_map)
print(result)

### Verification
expected = 3
assert result == expected, f"Expected {expected}, but got {result}"
print("Sample test passed!")

3
Sample test passed!


In [ ]:
### Dataset Test
dataset_filename = 'dataset_30266_3.txt' 

try:
    lines = get_dataset_lines(dataset_filename)
    if lines:
        spectrum, threshold, max_score = parse_spectral_dictionary_input(lines)
        aa_mass_map = get_aa_to_mass_table(toy=False)
        
        result = SizeOfSpectralDictionary(spectrum, threshold, max_score, aa_mass_map)
        print(result)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")

343


# Probability of Spectral Dictionary Problem

We can now compute the probability of a spectral dictionary as:

$$Pr(Dictionary_{threshold}(Spectrum')) = \sum_{t \ge threshold} Pr(m, t)$$

**Code Challenge**: Solve the Probability of Spectral Dictionary Problem.

**Input**: A spectral vector $Spectrum'$, an integer $threshold$, and an integer $max\_score$.

**Output**: The probability of the dictionary $Dictionary_{threshold}(Spectrum')$.

Note: Use the provided $max\_score$ for the height of your table.

**Extra Dataset**

**Sample Input**:

```
4 -3 -2 3 3 -4 5 -3 -1 -1 3 4 1 3
1
8
```

**Sample Output**:

```
0.375
```

In [ ]:
from collections import defaultdict

def ProbabilityOfSpectralDictionary(spectrum, threshold, max_score, aa_mass_map):
    # Computes the probability of generating a peptide with score >= threshold
    m = len(spectrum)
    # dp[i][t] = probability of peptides of mass i having score t
    dp = [defaultdict(float) for _ in range(m + 1)]
    
    # Base case: mass 0 has score 0 with probability 1.0
    dp[0][0] = 1.0
    
    aa_masses = list(aa_mass_map.values())
    prob_aa = 1.0 / len(aa_masses)
    
    for i in range(1, m + 1):
        for w in aa_masses:
            if i >= w:
                prev_mass = i - w
                current_score_gain = spectrum[i-1]
                
                for prev_score, prev_prob in dp[prev_mass].items():
                    new_score = prev_score + current_score_gain
                    
                    # Track scores only up to max_score
                    if new_score <= max_score:
                        dp[i][new_score] += prev_prob * prob_aa
                    
    # Sum probabilities for peptides of full mass m with score >= threshold
    result = 0.0
    for score, prob in dp[m].items():
        if score >= threshold:
            result += prob
            
    return result

In [ ]:
### Sample Input Execution
sample_input = """
4 -3 -2 3 3 -4 5 -3 -1 -1 3 4 1 3
1
8
""".strip().split('\n')

spectrum, threshold, max_score = parse_spectral_dictionary_input(sample_input)

# For the sample, the alphabet consists only of X and Z.
aa_mass_map = {'X': 4, 'Z': 5}

result = ProbabilityOfSpectralDictionary(spectrum, threshold, max_score, aa_mass_map)
print(result)

### Verification
expected = 0.375
assert abs(result - expected) < 1e-6, f"Expected {expected}, but got {result}"
print("Sample test passed!")

0.375
Sample test passed!


In [ ]:
### Dataset Test
dataset_filename = 'dataset_30266_8.txt' 

try:
    lines = get_dataset_lines(dataset_filename)
    if lines:
        spectrum, threshold, max_score = parse_spectral_dictionary_input(lines)
        aa_mass_map = get_aa_to_mass_table(toy=False)
        
        result = ProbabilityOfSpectralDictionary(spectrum, threshold, max_score, aa_mass_map)
        print(result)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")

2.0740625000000008e-05


# Spectral Alignment Problem

Although the recurrence from the previous step computes the score of a modified peptide solving the Spectral Alignment Problem, we also need to reconstruct this peptide. In order to achieve this goal, we will need to implement a backtracking approach similar to the backtracking approach described in sequence alignment, which we leave to you as an exercise.

**Code Challenge**: Solve the Spectral Alignment Problem.

**Input**: A peptide $Peptide$, a spectral vector $Spectrum'$, and an integer $k$.

**Output**: A peptide $Peptide'$ related to $Peptide$ by up to $k$ modifications with maximal score against $Spectrum'$ out of all possibilities.

**Sample Input**:

```
XXZ
4 -3 -2 3 3 -4 5 -3 -1 -1 3 4 1 -1
2
```

**Sample Output**:

```
XX(-1)Z(+2)
```

In [ ]:
def SpectralAlignment(peptide, spectrum, k, aa_mass_map):
    # Finds the optimal alignment between a peptide and spectrum with up to k modifications
    n = len(peptide)
    m = len(spectrum)
    
    # DP[i][j][t] = max score aligning first i amino acids to mass j with t modifications
    dp = [[[-float('inf')] * (k + 1) for _ in range(m + 1)] for _ in range(n + 1)]
    
    # Backtrack[i][j][t] stores (prev_j, prev_t) to reconstruct the solution
    backtrack = [[[None] * (k + 1) for _ in range(m + 1)] for _ in range(n + 1)]
    
    dp[0][0][0] = 0
    
    for i in range(1, n + 1):
        aa = peptide[i-1]
        w = aa_mass_map[aa]
        
        for t in range(k + 1):
            
            # Optimization: Find best previous position for a modification transition
            best_prev_val = -float('inf')
            best_prev_pos = -1
            
            for j in range(1, m + 1):
                if t > 0:
                    prev_p = j - 1
                    val = dp[i-1][prev_p][t-1]
                    if val > best_prev_val:
                        best_prev_val = val
                        best_prev_pos = prev_p
                
                score_current = spectrum[j-1]
                
                # 1. Transition with no modification (standard amino acid match)
                score_unmod = -float('inf')
                prev_j_unmod = j - w
                if prev_j_unmod >= 0:
                    if dp[i-1][prev_j_unmod][t] > -float('inf'):
                        score_unmod = dp[i-1][prev_j_unmod][t] + score_current
                        
                # 2. Transition with modification (from best previous position)
                score_mod = -float('inf')
                if t > 0 and best_prev_val > -float('inf'):
                    score_mod = best_prev_val + score_current
                
                # Choose the transition that yields the higher score
                if score_unmod >= score_mod and score_unmod > -float('inf'):
                    dp[i][j][t] = score_unmod
                    backtrack[i][j][t] = (prev_j_unmod, t)
                elif score_mod > score_unmod: # score_mod must be > -inf
                    dp[i][j][t] = score_mod
                    backtrack[i][j][t] = (best_prev_pos, t-1)
                    
    # Find the maximum score for the full peptide and full spectrum with any t <= k
    max_score = -float('inf')
    best_t = -1
    
    for t in range(k + 1):
        if dp[n][m][t] > max_score:
            max_score = dp[n][m][t]
            best_t = t
            
    # Backtrack to reconstruct the aligned peptide
    if best_t == -1:
        return "No solution found"
        
    result_parts = []
    curr_i, curr_j, curr_t = n, m, best_t
    
    while curr_i > 0:
        prev_j, prev_t = backtrack[curr_i][curr_j][curr_t]
        
        aa = peptide[curr_i-1]
        standard_mass = aa_mass_map[aa]
        actual_mass_diff = curr_j - prev_j
        delta = actual_mass_diff - standard_mass
        
        if delta == 0:
            result_parts.append(aa)
        else:
            result_parts.append(f"{aa}({delta:+d})")
            
        curr_i -= 1
        curr_j = prev_j
        curr_t = prev_t
        
    return "".join(reversed(result_parts))

In [ ]:
### Sample Input Execution
sample_input = """
XXZ
4 -3 -2 3 3 -4 5 -3 -1 -1 3 4 1 -1
2
""".strip().split('\n')

peptide, spectrum, k = parse_spectral_alignment_input(sample_input)

aa_mass_map = get_aa_to_mass_table(toy=True)

result = SpectralAlignment(peptide, spectrum, k, aa_mass_map)
print(result)

### Verification
expected = "XX(-1)Z(+2)"
assert result == expected, f"Expected {expected}, but got {result}"
print("Sample test passed!")

XX(-1)Z(+2)
Sample test passed!


In [ ]:
### Dataset Test
dataset_filename = 'dataset_30269_3.txt'

try:
    lines = get_dataset_lines(dataset_filename)
    if lines:
        peptide, spectrum, k = parse_spectral_alignment_input(lines)
        aa_mass_map = get_aa_to_mass_table(toy=False)
        
        result = SpectralAlignment(peptide, spectrum, k, aa_mass_map)
        print(result)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    import traceback
    traceback.print_exc()

S(-38)WRYVP(+38)
